# Shanta New Luika Excel Extractor

## Overview

This notebook extracts structured daily leaching data from a source Excel workbook using a row/column mapping configuration, performs cleansing and field derivation, runs a preflight validation step, and can optionally upload the resulting records to the LeachIT API.

The logic in this notebook has been adapted from the original desktop upload utility. The original program used a Tkinter user interface and read its mapping from a YAML settings file. Here, the same workflow is exposed in notebook form so each step can be inspected, tested, and run interactively.

In [96]:
# Core imports
import math
import json
from datetime import datetime
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, Mapping, Optional, Sequence, cast, List, Tuple

import numpy as np
import pandas as pd
import yaml
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

## Notebook configuration

This cell defines the workbook path, output options, API details, and runtime switches. In the desktop program, these values were entered through the UI and optionally persisted to a user settings file. In the notebook, they are controlled directly here.

In [97]:
# === User-configurable notebook settings ===

# Source workbook
XLSX_PATH = Path("NLGM-Met Accounting 202502.xlsx")

# Optional CSV export
SAVE_CSV = False
CSV_OUTPUT_PATH = Path(f"extract_{datetime.now():%Y%m}.csv")

# API settings
BASE_URL = "http://localhost:8000/api"  # https://leachit-poc.dev.iesim.biz/api
ADMIN_TOKEN = "Q3ljbGljLUZhY3RzaGVldDktUGF5ZXI="
CUSTOMER = "shanta"
SITE = "new_luika"

BATCH_SIZE = 5000
DRY_RUN = True           # True = do everything except upload

# Logging helper for notebook
def log(msg: str) -> None:
	print(msg)

## Embedded YAML extraction config

The original tool reads a YAML file named `settings_default.yaml` to determine which workbook sheets, rows, and columns should be extracted, along with UI defaults. In the notebook, that YAML is embedded directly as a string so the entire workflow is self-contained.

In [98]:
CONFIG_YAML_TEXT = r"""
sheets:
  input:
    name: "Input"
    date_row: 4
    first_data_col: "E"

    fields:
      - { row: 53, name: "Percent_Solids" }
      - { row: 63, name: "WAD_CN_Tailings_ppm" }

      - { row: 71, name: "Ph_Leach_Tank_01" }
      - { row: 72, name: "Ph_Leach_Tank_02" }
      - { row: 73, name: "Ph_Cil_Tank_01" }
      - { row: 74, name: "Ph_Cil_Tank_02" }
      - { row: 75, name: "Ph_Cil_Tank_03" }
      - { row: 76, name: "Ph_Cil_Tank_04" }
      - { row: 77, name: "Ph_Cil_Tank_05" }
      - { row: 78, name: "Ph_Cil_Tank_06" }
      - { row: 79, name: "Ph_Cil_Tank_07" }
      - { row: 80, name: "Ph_Cil_Tank_08" }
      - { row: 81, name: "Ph_Cil_Tank_09" }

      - { row: 83, name: "Free_CN_Leach_Tank_1_ppm" }
      - { row: 84, name: "Free_CN_Leach_Tank_2_ppm" }
      - { row: 85, name: "Free_CN_Cil_Tank_1_ppm" }
      - { row: 86, name: "Free_CN_Cil_Tank_2_ppm" }
      - { row: 87, name: "Free_CN_Cil_Tank_3_ppm" }
      - { row: 88, name: "Free_CN_Cil_Tank_4_ppm" }
      - { row: 89, name: "Free_CN_Cil_Tank_5_ppm" }
      - { row: 90, name: "Free_CN_Cil_Tank_6_ppm" }
      - { row: 91, name: "Free_CN_Cil_Tank_7_ppm" }
      - { row: 92, name: "Free_CN_Cil_Tank_8_ppm" }
      - { row: 93, name: "Free_CN_Cil_Tank_9_ppm" }

      - { row: 95, name: "Do_Leach_Tank_1_ppm" }
      - { row: 96, name: "Do_Leach_Tank_2_ppm" }
      - { row: 97, name: "Do_Cil_Tank_1_ppm" }
      - { row: 98, name: "Do_Cil_Tank_2_ppm" }
      - { row: 99, name: "Do_Cil_Tank_3_ppm" }
      - { row: 100, name: "Do_Cil_Tank_4_ppm" }
      - { row: 101, name: "Do_Cil_Tank_5_ppm" }
      - { row: 102, name: "Do_Cil_Tank_6_ppm" }
      - { row: 103, name: "Do_Cil_Tank_7_ppm" }
      - { row: 104, name: "Do_Cil_Tank_8_ppm" }
      - { row: 105, name: "Do_Cil_Tank_9_ppm" }

      - { row: 107, name: "Carbon_Concentration_Cil_Tank_1_gpl" }
      - { row: 108, name: "Carbon_Concentration_Cil_Tank_2_gpl" }
      - { row: 109, name: "Carbon_Concentration_Cil_Tank_3_gpl" }
      - { row: 110, name: "Carbon_Concentration_Cil_Tank_4_gpl" }
      - { row: 111, name: "Carbon_Concentration_Cil_Tank_5_gpl" }
      - { row: 112, name: "Carbon_Concentration_Cil_Tank_6_gpl" }
      - { row: 113, name: "Carbon_Concentration_Cil_Tank_7_gpl" }
      - { row: 114, name: "Carbon_Concentration_Cil_Tank_8_gpl" }
      - { row: 115, name: "Carbon_Concentration_Cil_Tank_9_gpl" }

      - { row: 125, name: "Leach_Feed_Grade_Au_Gt_Day" }
      - { row: 128, name: "Leach_Tailings_Grade_Au_Gpt" }
      - { row: 131, name: "Leach_Feed_Grade_Ag_Gt_Day" }
      - { row: 134, name: "Leach_Tailings_Grade_Ag_Gpt" }

      - { row: 151, name: "Au_Undissolved_Leach_Tank_1_gpt" }
      - { row: 152, name: "Au_Undissolved_Leach_Tank_2_gpt" }
      - { row: 153, name: "Au_Undissolved_Cil_Tank_1_gpt" }
      - { row: 154, name: "Au_Undissolved_Cil_Tank_2_gpt" }
      - { row: 155, name: "Au_Undissolved_Cil_Tank_3_gpt" }
      - { row: 156, name: "Au_Undissolved_Cil_Tank_4_gpt" }
      - { row: 157, name: "Au_Undissolved_Cil_Tank_5_gpt" }
      - { row: 158, name: "Au_Undissolved_Cil_Tank_6_gpt" }
      - { row: 159, name: "Au_Undissolved_Cil_Tank_7_gpt" }
      - { row: 160, name: "Au_Undissolved_Cil_Tank_8_gpt" }
      - { row: 161, name: "Au_Undissolved_Cil_Tank_9_gpt" }

      - { row: 162, name: "Ag_Undissolved_Leach_Tank_1_gpt" }
      - { row: 163, name: "Ag_Undissolved_Leach_Tank_2_gpt" }
      - { row: 164, name: "Ag_Undissolved_Cil_Tank_1_gpt" }
      - { row: 165, name: "Ag_Undissolved_Cil_Tank_2_gpt" }
      - { row: 166, name: "Ag_Undissolved_Cil_Tank_3_gpt" }
      - { row: 167, name: "Ag_Undissolved_Cil_Tank_4_gpt" }
      - { row: 168, name: "Ag_Undissolved_Cil_Tank_5_gpt" }
      - { row: 169, name: "Ag_Undissolved_Cil_Tank_6_gpt" }
      - { row: 170, name: "Ag_Undissolved_Cil_Tank_7_gpt" }
      - { row: 171, name: "Ag_Undissolved_Cil_Tank_8_gpt" }

      - { row: 173, name: "Au_Dissolved_Leach_Tank_1_gpt" }
      - { row: 174, name: "Au_Dissolved_Leach_Tank_2_gpt" }
      - { row: 175, name: "Au_Dissolved_Cil_Tank_1_gpt" }
      - { row: 176, name: "Au_Dissolved_Cil_Tank_2_gpt" }
      - { row: 177, name: "Au_Dissolved_Cil_Tank_3_gpt" }
      - { row: 178, name: "Au_Dissolved_Cil_Tank_4_gpt" }
      - { row: 179, name: "Au_Dissolved_Cil_Tank_5_gpt" }
      - { row: 180, name: "Au_Dissolved_Cil_Tank_6_gpt" }
      - { row: 181, name: "Au_Dissolved_Cil_Tank_7_gpt" }
      - { row: 182, name: "Au_Dissolved_Cil_Tank_8_gpt" }
      - { row: 183, name: "Au_Dissolved_Cil_Tank_9_gpt" }

      - { row: 184, name: "Ag_Dissolved_Leach_Tank_1_gpt" }
      - { row: 185, name: "Ag_Dissolved_Leach_Tank_2_gpt" }
      - { row: 186, name: "Ag_Dissolved_Cil_Tank_1_gpt" }
      - { row: 187, name: "Ag_Dissolved_Cil_Tank_2_gpt" }
      - { row: 188, name: "Ag_Dissolved_Cil_Tank_3_gpt" }
      - { row: 189, name: "Ag_Dissolved_Cil_Tank_4_gpt" }
      - { row: 190, name: "Ag_Dissolved_Cil_Tank_5_gpt" }
      - { row: 191, name: "Ag_Dissolved_Cil_Tank_6_gpt" }
      - { row: 192, name: "Ag_Dissolved_Cil_Tank_7_gpt" }
      - { row: 193, name: "Ag_Dissolved_Cil_Tank_8_gpt" }

      - { row: 195, name: "Au_Loading_Cil_Tank_1_gptc" }
      - { row: 196, name: "Au_Loading_Cil_Tank_2_gptc" }
      - { row: 197, name: "Au_Loading_Cil_Tank_3_gptc" }
      - { row: 198, name: "Au_Loading_Cil_Tank_4_gptc" }
      - { row: 199, name: "Au_Loading_Cil_Tank_5_gptc" }
      - { row: 200, name: "Au_Loading_Cil_Tank_6_gptc" }
      - { row: 201, name: "Au_Loading_Cil_Tank_7_gptc" }
      - { row: 202, name: "Au_Loading_Cil_Tank_8_gptc" }
      - { row: 203, name: "Au_Loading_Cil_Tank_9_gptc" }

      - { row: 204, name: "Ag_Loading_Cil_Tank_1_gptc" }
      - { row: 205, name: "Ag_Loading_Cil_Tank_2_gptc" }
      - { row: 206, name: "Ag_Loading_Cil_Tank_3_gptc" }
      - { row: 207, name: "Ag_Loading_Cil_Tank_4_gptc" }
      - { row: 208, name: "Ag_Loading_Cil_Tank_5_gptc" }
      - { row: 209, name: "Ag_Loading_Cil_Tank_6_gptc" }
      - { row: 210, name: "Ag_Loading_Cil_Tank_7_gptc" }
      - { row: 211, name: "Ag_Loading_Cil_Tank_8_gptc" }
      - { row: 212, name: "Ag_Loading_Cil_Tank_9_gptc" }

      - { row: 234, name: "Leach_Feed_Gt_150um_Day" }
      - { row: 235, name: "Leach_Feed_Gt_75um_Day" }
      - { row: 236, name: "Leach_Feed_Lt_75um_Day" }
      - { row: 249, name: "NaCN_Withdrawn_Tpd" }

  calcs:
    name: "Calcs"
    date_row: 4
    first_data_col: "E"

    fields:
      - { row: 34, name: "Leach_Feed_Dry_t" }
      - { row: 59, name: "Au_CIL_Solids_g" }
      - { row: 65, name: "Au_Tailings_Solids_g" }
      - { row: 61, name: "Au_Leach_Feed_g" }
      - { row: 64, name: "Ag_Leach_Feed_g" }
      - { row: 67, name: "Au_Tailings_g" }
      - { row: 70, name: "Ag_Cil_Tailings_g" }
      - { row: 72, name: "Au_Leach_Feed_Grade_gpt" }
      - { row: 78, name: "Au_Cil_Tailings_Residuals_gpt" }
      - { row: 138, name: "Recovery_pct" }

dedupe_suffix: "_dup"
fallback_label_prefix: "Row-"

ui_defaults:
  base: "https://leachit-poc.dev.iesim.biz/api"
  token: ""
  customer: "shanta"
  site: "new_luika"
  tag: "bulk_upload"
  mode: "append"
  batch: 5000
  dry_run: true
  save_csv: false
"""

## Excel extraction helpers

These helper functions implement the workbook extraction layer. They convert Excel column letters to zero-based indices, expand row ranges, read sheets, collect field definitions, deduplicate labels, and extract day-wise column blocks into tabular form. This is the core of the workbook-to-DataFrame transformation.

In [99]:
def excel_col_to_idx(col_letter: str) -> int:
	col_letter = col_letter.strip().upper()
	n = 0
	for c in col_letter:
		n = n * 26 + (ord(c) - ord("A") + 1)
	return n - 1


def expand_ranges(rows_spec: Sequence[int | Sequence[int]]) -> list[int]:
	"""Accepts [ints or [start,end]] -> explicit 1-based ints."""
	out: list[int] = []
	for item in rows_spec:
		if isinstance(item, Sequence) and not isinstance(item, (str, bytes)) and len(item) == 2:
			start = int(item[0])
			end = int(item[1])
			out.extend(range(start, end + 1))
		elif isinstance(item, int):
			out.append(item)
		else:
			raise TypeError(f"Invalid rows_spec item: {item}")
	return out


def read_sheet(path: Path, sheet_name: str) -> pd.DataFrame:
	return pd.read_excel(path, sheet_name=sheet_name, header=None, engine="openpyxl")


def _collect_fields_from_rows(
	df: pd.DataFrame,
	rows_1based: List[int],
	label_overrides: Optional[Dict[int, str]],
	fallback_prefix: str,
) -> List[Tuple[str, int]]:
	fields: List[Tuple[str, int]] = []
	for r in rows_1based:
		row_idx0 = r - 1
		raw_label: Any = df.iat[row_idx0, 1]

		if raw_label is None:
			label = f"{fallback_prefix}{r}"
		elif isinstance(raw_label, float) and pd.isna(raw_label):
			label = f"{fallback_prefix}{r}"
		else:
			label = str(raw_label).strip()
			if label == "":
				label = f"{fallback_prefix}{r}"

		if label_overrides and r in label_overrides:
			label = label_overrides[r].strip()

		fields.append((label, r))
	return fields


def _collect_fields_from_explicit(explicit_fields: Sequence[Mapping[str, Any]]) -> list[tuple[str, int]]:
	out: list[tuple[str, int]] = []
	for item in explicit_fields:
		r = int(item["row"])
		name = str(item["name"]).strip()
		out.append((name, r))
	return out


def _dedupe_labels(labels: Sequence[str], suffix: str) -> list[str]:
	seen: dict[str, int] = {}
	final: list[str] = []
	for lab in labels:
		if lab not in seen:
			seen[lab] = 1
			final.append(lab)
		else:
			seen[lab] += 1
			final.append(f"{lab}{suffix}")
	return final


def _extract_sheet_columns(
	df: pd.DataFrame,
	first_data_col_letter: str,
	n_days: int,
	fields_lr: Sequence[tuple[str, int]],
) -> pd.DataFrame:
	start_col_idx = excel_col_to_idx(first_data_col_letter)
	row_idx0 = [r - 1 for _, r in fields_lr]
	labels = [lab for lab, _ in fields_lr]

	block = df.iloc[row_idx0, start_col_idx : start_col_idx + n_days]
	arr = block.to_numpy().T

	out = pd.DataFrame(arr, columns=labels).reset_index(drop=True)
	if len(out) != n_days:
		out = out.reindex(range(n_days)).reset_index(drop=True)
	return out


def build_extract(xlsx_path: Path, config_yaml_text: str) -> pd.DataFrame:
	cfg = cast(dict[str, Any], yaml.safe_load(config_yaml_text))
	sheets = cast(dict[str, Any], cfg["sheets"])
	sh_input = cast(dict[str, Any], sheets["input"])
	sh_calcs = cast(dict[str, Any], sheets["calcs"])

	df_input = read_sheet(xlsx_path, str(sh_input["name"]))
	df_calcs = read_sheet(xlsx_path, str(sh_calcs["name"]))

	date_row_idx0 = int(sh_input["date_row"]) - 1
	start_col_idx = excel_col_to_idx(str(sh_input["first_data_col"]))
	date_series = df_input.iloc[date_row_idx0, start_col_idx:]
	date_series = date_series.dropna(how="all")
	dates = pd.to_datetime(date_series, errors="coerce")
	n_days = len(dates)

	input_fields_cfg = sh_input.get("fields")
	input_label_overrides = {int(k): str(v) for k, v in cast(dict[Any, Any], sh_input.get("label_overrides") or {}).items()}
	if input_fields_cfg:
		input_fields = _collect_fields_from_explicit(cast(Sequence[Mapping[str, Any]], input_fields_cfg))
	else:
		rows_spec = cast(Sequence[int | Sequence[int]], sh_input["rows"])
		input_rows = expand_ranges(rows_spec)
		input_fields = _collect_fields_from_rows(
			df_input,
			input_rows,
			input_label_overrides,
			str(cfg["fallback_label_prefix"]),
		)

	calcs_fields_cfg = sh_calcs.get("fields")
	calcs_label_overrides = {int(k): str(v) for k, v in cast(dict[Any, Any], sh_calcs.get("label_overrides") or {}).items()}
	if calcs_fields_cfg:
		calcs_fields = _collect_fields_from_explicit(cast(Sequence[Mapping[str, Any]], calcs_fields_cfg))
	else:
		rows_spec = cast(Sequence[int | Sequence[int]], sh_calcs["rows"])
		calcs_rows = expand_ranges(rows_spec)
		calcs_fields = _collect_fields_from_rows(
			df_calcs,
			calcs_rows,
			calcs_label_overrides,
			str(cfg["fallback_label_prefix"]),
		)

	all_labels = [lab for lab, _ in input_fields] + [lab for lab, _ in calcs_fields]
	deduped = _dedupe_labels(all_labels, str(cfg.get("dedupe_suffix", "_dup")))
	dedup_input = deduped[: len(input_fields)]
	dedup_calcs = deduped[len(input_fields) :]

	out = pd.DataFrame({"Date": dates.values})
	df_input_vals = _extract_sheet_columns(
		df_input,
		str(sh_input["first_data_col"]),
		n_days,
		list(zip(dedup_input, [r for _, r in input_fields])),
	)
	df_calcs_vals = _extract_sheet_columns(
		df_calcs,
		str(sh_calcs["first_data_col"]),
		n_days,
		list(zip(dedup_calcs, [r for _, r in calcs_fields])),
	)
	out = pd.concat([out, df_input_vals, df_calcs_vals], axis=1).copy()
	return out

## Cleansing, derivation, and JSON conversion helpers

These functions prepare the extracted table for downstream use. They standardise column names, clean invalid values, derive the required upload fields, normalise particle size fractions, and convert the DataFrame into JSON-safe row records for the API payload.

In [100]:
REQUIRED = [
	"date",
	"cn",
	"do",
	"grade",
	"percent_solids",
	"throughput",
	"coarse",
	"fines",
	"ultrafines"
	]


def json_safe(v: Any) -> Any:
	if v is None:
		return None
	try:
		if pd.isna(v):
			return None
	except Exception:
		pass
	if isinstance(v, np.generic):
		v = v.item()
	if isinstance(v, float):
		if math.isnan(v) or math.isinf(v):
			return None
		return v
	if isinstance(v, (int, bool)):
		return v
	if isinstance(v, (pd.Timestamp, datetime)):
		return v.date().isoformat()
	if isinstance(v, np.datetime64):
		try:
			return pd.to_datetime(v).date().isoformat()
		except Exception:
			return None
	if isinstance(v, str):
		s = v.strip()
		if s.lower() in {"nan", "na", "none", ""}:
			return None
		return s
	if isinstance(v, (list, tuple)):
		return [json_safe(x) for x in v]
	if isinstance(v, dict):
		return {str(k): json_safe(val) for k, val in v.items()}
	return str(v)


def df_to_json_records(df: pd.DataFrame) -> list[dict[str, Any]]:
	obj = df.copy()
	if "date" in obj.columns:
		obj["date"] = pd.to_datetime(obj["date"], errors="coerce")
	obj = obj.astype(object)
	records_raw = obj.to_dict(orient="records")
	records = cast(list[dict[Any, Any]], records_raw)
	return [{str(k): json_safe(v) for k, v in row.items()} for row in records]


def clean_like_server(name: str) -> str:
	s = name.strip()
	s = s.replace("%", "pct").replace("/", " ").replace("-", " ")
	s = s.replace("(", " ").replace(")", " ")
	s = "_".join(s.split()).lower()
	return s


def cleanse_df(df: pd.DataFrame) -> pd.DataFrame:
	out = df.copy()
	out.columns = [str(c).strip() for c in out.columns]

	if "date" in out.columns:
		out["date"] = pd.to_datetime(out["date"], errors="coerce")
		out = out[out["date"].notna()].copy()

	num_cols = out.select_dtypes(include=[np.number]).columns
	if len(num_cols):
		out.loc[:, num_cols] = out.loc[:, num_cols].replace([np.inf, -np.inf], np.nan)

	return out


def _series_of_nan(index: pd.Index) -> pd.Series:
	return pd.Series(np.nan, index=index, dtype="float64")


def _coerce_series(value: Any, index: pd.Index) -> pd.Series:
	if isinstance(value, pd.Series):
		return value.reindex(index)
	if isinstance(value, np.ndarray):
		return pd.Series(value, index=index)
	if isinstance(value, list):
		return pd.Series(value, index=index[: len(value)])
	return _series_of_nan(index)


def prefer_first(df: pd.DataFrame, candidates: Sequence[str]) -> pd.Series:
	for c in candidates:
		if c in df.columns:
			s = pd.to_numeric(df[c], errors="coerce")
			if s.notna().any():
				return s
	return _series_of_nan(df.index)


def normalise_triplet(coarse: pd.Series, fines: pd.Series, ultra: pd.Series) -> tuple[pd.Series, pd.Series, pd.Series]:
	trip = pd.concat([coarse, fines, ultra], axis=1)
	trip.columns = ["coarse", "fines", "ultrafines"]
	for col in trip.columns:
		trip[col] = pd.to_numeric(trip[col], errors="coerce").clip(lower=0)
	s = trip.sum(axis=1)
	mask = s.between(95, 105)
	trip.loc[mask, ["coarse", "fines", "ultrafines"]] = (
		trip.loc[mask, ["coarse", "fines", "ultrafines"]].div(s[mask], axis=0) * 100.0
	)
	return trip["coarse"], trip["fines"], trip["ultrafines"]


def derive_required_fields(df: pd.DataFrame) -> pd.DataFrame:
	g = df
	out = pd.DataFrame(index=g.index)

	date_input = _coerce_series(g["date"], g.index) if "date" in g.columns else _series_of_nan(g.index)
	out["date"] = pd.to_datetime(date_input, errors="coerce")

	percent_solids_input = g["percent_solids"] if "percent_solids" in g.columns else _series_of_nan(g.index)
	out["percent_solids"] = pd.to_numeric(percent_solids_input, errors="coerce")

	out["cn"] = prefer_first(
		g,
		[
			"avg_free_cn_tank_ppm",
			"avg_free_cn_leach_ppm",
			"free_cn_leach_tank_1_ppm",
			"cyanide_profile_ppm_leach_tank_1",
			"free_cn_leach_tank_2_ppm",
		],
	)

	out["do"] = prefer_first(
		g,
		[
			"avg_do_tank_ppm",
			"do_leach_tank_1_ppm_imputed",
			"do_leach_tank_1_ppm",
			"do_leach_tank_2_ppm",
		],
	)

	out["grade"] = prefer_first(
		g,
		[
			"au_feed_grade_ppm",
			"leach_feed_grade_au_gt_day",
		],
	)

	out["throughput"] = prefer_first(
		g,
		[
			"leach_feed_dry_t",
			"leach_feed_throughput_m3day",
		],
	)

	if "pct_lt_75" in g.columns and "pct_lt_150" in g.columns:
		pct_lt_75 = pd.to_numeric(g["pct_lt_75"], errors="coerce")
		pct_lt_150 = pd.to_numeric(g["pct_lt_150"], errors="coerce")
		ultrafines = pct_lt_75 * 100.0
		fines = (pct_lt_150 - pct_lt_75) * 100.0
		coarse = (1.0 - pct_lt_150) * 100.0
	else:
		lt_75_src = g["leach_feed_lt_75um_day"] if "leach_feed_lt_75um_day" in g.columns else _series_of_nan(g.index)
		gt_75_src = g["leach_feed_gt_75um_day"] if "leach_feed_gt_75um_day" in g.columns else _series_of_nan(g.index)
		gt_150_src = g["leach_feed_gt_150um_day"] if "leach_feed_gt_150um_day" in g.columns else _series_of_nan(g.index)

		lt_75 = pd.to_numeric(lt_75_src, errors="coerce")
		gt_75 = pd.to_numeric(gt_75_src, errors="coerce")
		gt_150 = pd.to_numeric(gt_150_src, errors="coerce")

		ultrafines = lt_75
		fines = gt_75 - gt_150
		coarse = gt_150

	coarse, fines, ultrafines = normalise_triplet(coarse, fines, ultrafines)
	out["coarse"] = coarse
	out["fines"] = fines
	out["ultrafines"] = ultrafines

	for c in ["percent_solids", "cn", "do", "grade", "throughput", "coarse", "fines", "ultrafines"]:
		out[c] = pd.to_numeric(out[c], errors="coerce").round(6)

	return out


def preflight(df_raw: pd.DataFrame, log: Callable[[str], None]) -> pd.DataFrame:
	orig_cols = list(df_raw.columns)
	cols = [clean_like_server(str(c)) for c in orig_cols]
	df = df_raw.copy()
	df.columns = cols

	log("== Preflight: source ==")
	log(f"Original columns ({len(orig_cols)}): {orig_cols[:10]}{' ...' if len(orig_cols) > 10 else ''}")
	log(f"Cleaned  columns ({len(cols)}): {cols[:10]}{' ...' if len(cols) > 10 else ''}")

	if "date" in df.columns:
		dt = pd.to_datetime(df["date"], errors="coerce")
		bad = int(dt.isna().sum())
		log(f"Date parse: {len(dt) - bad} ok, {bad} bad")

	derived = derive_required_fields(df)
	missing_by_row = derived[REQUIRED].isna().any(axis=1)
	n_missing = int(missing_by_row.sum())
	log("\n== Derived REQUIRED snapshot (first 3 rows) ==")
	log(derived.head(3).to_string())
	log(f"\nRows missing any REQUIRED fields: {n_missing} / {len(derived)}")
	if n_missing:
		sample = derived[missing_by_row].head(3)
		log("\nSample rows with missing REQUIRED fields:")
		log(sample.to_string())

	return df

## API session and upload helpers

These functions prepare a retry-enabled HTTP session and upload historical records to the configured API endpoint in batches. This preserves the behaviour of the original application while making the upload step explicit and inspectable in the notebook.

In [101]:
def make_session_with_retry(
	total: int = 3,
	backoff: float = 0.5,
	status_forcelist: Iterable[int] = (502, 503, 504),
) -> requests.Session:
	session = requests.Session()
	retries = Retry(
		total=total,
		backoff_factor=backoff,
		status_forcelist=tuple(status_forcelist),
		allowed_methods=frozenset(["POST"]),
	)
	adapter = HTTPAdapter(max_retries=retries)
	session.mount("http://", adapter)
	session.mount("https://", adapter)
	return session


def post_rows(
	session: requests.Session,
	base: str,
	headers: dict[str, str],
	customer: str,
	site: str,
	tag: str,
	row_batch: list[dict[str, Any]],
	req_mode: str,
) -> dict[str, Any]:
	payload = {
		"customer": customer,
		"site": site,
		"mode": req_mode,
		"tag": tag,
		"rows": row_batch,
	}
	json.dumps(payload, allow_nan=False)
	r = session.post(f"{base}/ingest/historical", json=payload, headers=headers, timeout=300)
	if not r.ok:
		try:
			msg = r.json()
		except Exception:
			msg = r.text
		raise RuntimeError(f"Upload failed {r.status_code}: {msg}")
	return cast(dict[str, Any], r.json())

## Extract data from the workbook

This cell performs the workbook extraction step. It reads the Input and Calcs sheets, applies the YAML mapping, and returns a wide table with one row per day and one column per extracted field.

In [102]:
df_extract = build_extract(XLSX_PATH, CONFIG_YAML_TEXT)
df_extract.columns = [c if c != "Date" else "date" for c in df_extract.columns]

log(f"Extracted shape: {df_extract.shape}")
display(df_extract.head())

Extracted shape: (28, 123)


,date,Percent_Solids,WAD_CN_Tailings_ppm,Ph_Leach_Tank_01,Ph_Leach_Tank_02,Ph_Cil_Tank_01,Ph_Cil_Tank_02,Ph_Cil_Tank_03,Ph_Cil_Tank_04,Ph_Cil_Tank_05,...,Leach_Feed_Dry_t,Au_CIL_Solids_g,Au_Tailings_Solids_g,Au_Leach_Feed_g,Ag_Leach_Feed_g,Au_Tailings_g,Ag_Cil_Tailings_g,Au_Leach_Feed_Grade_gpt,Au_Cil_Tailings_Residuals_gpt,Recovery_pct
0,2026-02-01,50.333333,276,10.87,10.95,10.935,11.125,11.04,11.065,11.08,...,2304.2491,5737.580259,576.062275,5739.853988,11843.840374,578.336004,3133.778776,2.490987,0.25,89.959839
1,2026-02-02,51.375,240,11.07,11.155,11.155,11.355,11.255,11.255,11.17,...,2607.9614,6702.460798,625.910736,6704.92916,12779.01086,632.081642,3664.185767,2.570946,0.24,90.661479
2,2026-02-03,50.75,250,11.16,11.235,11.25,11.425,11.235,11.225,11.225,...,2668.0368,6630.071448,680.349384,6632.660627,13326.843816,682.938563,4095.436488,2.48597,0.255,89.738431
3,2026-02-04,51.5,195,11.26,11.36,11.3,11.52,11.24,11.12,11.14,...,1973.4025,4844.703137,522.951662,4845.632361,10172.889887,524.81011,2960.10375,2.455471,0.265,89.205703
4,2026-02-05,50.608696,184,11.115,11.185,11.105,11.3,11.18,11.16,11.24,...,2269.31625,5877.529088,601.368806,5879.743815,11766.404756,603.583534,3199.735913,2.590976,0.265,89.76834


##


## Clean and preflight the extracted data

This step standardises column names, removes invalid date rows, checks the required upload fields, and shows where any missing values remain after derivation. It is the safest place to inspect the data before saving or uploading.

In [103]:
df_wide = df_extract.copy()
df_wide.columns = [clean_like_server(str(c)) for c in df_wide.columns]
df_wide = cleanse_df(df_wide)

_ = preflight(df_wide, log)

== Preflight: source ==
Original columns (123): ['date', 'percent_solids', 'wad_cn_tailings_ppm', 'ph_leach_tank_01', 'ph_leach_tank_02', 'ph_cil_tank_01', 'ph_cil_tank_02', 'ph_cil_tank_03', 'ph_cil_tank_04', 'ph_cil_tank_05'] ...
Cleaned  columns (123): ['date', 'percent_solids', 'wad_cn_tailings_ppm', 'ph_leach_tank_01', 'ph_leach_tank_02', 'ph_cil_tank_01', 'ph_cil_tank_02', 'ph_cil_tank_03', 'ph_cil_tank_04', 'ph_cil_tank_05'] ...
Date parse: 28 ok, 0 bad

== Derived REQUIRED snapshot (first 3 rows) ==
        date  percent_solids   cn     do  grade  throughput    coarse      fines  ultrafines
0 2026-02-01       50.333333  380  22.00  2.490   2304.2491  3.957465  14.709277   81.333258
1 2026-02-02       51.375000  375  21.95  2.570   2607.9614  4.731759  14.618925   80.649316
2 2026-02-03       50.750000  350  22.00  2.485   2668.0368  7.352478  11.199077   74.095967

Rows missing any REQUIRED fields: 0 / 28


## Optional CSV export

If required, the extracted and cleaned wide table can be written to CSV before any upload occurs. This is useful for QA, offline review, or debugging workbook mappings.

In [104]:
if SAVE_CSV:
	CSV_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
	df_wide.to_csv(CSV_OUTPUT_PATH, index=False)
	log(f"Saved extracted CSV -> {CSV_OUTPUT_PATH}")

## Convert the cleaned table to upload records

The historical upload endpoint expects a JSON payload containing row records. This cell converts the cleaned DataFrame into JSON-safe dictionaries and shows a small sample before upload.

In [105]:
rows = df_to_json_records(df_wide)

log(f"Prepared {len(rows)} upload rows")
rows[:2]

Prepared 28 upload rows


[{'date': '2026-02-01',
  'percent_solids': 50.333333333333336,
  'wad_cn_tailings_ppm': 276,
  'ph_leach_tank_01': 10.870000000000001,
  'ph_leach_tank_02': 10.95,
  'ph_cil_tank_01': 10.934999999999999,
  'ph_cil_tank_02': 11.125,
  'ph_cil_tank_03': 11.04,
  'ph_cil_tank_04': 11.065,
  'ph_cil_tank_05': 11.079999999999998,
  'ph_cil_tank_06': 11.125,
  'ph_cil_tank_07': 11.245,
  'ph_cil_tank_08': 11.265,
  'ph_cil_tank_09': 11.26,
  'free_cn_leach_tank_1_ppm': 380,
  'free_cn_leach_tank_2_ppm': 360,
  'free_cn_cil_tank_1_ppm': 365,
  'free_cn_cil_tank_2_ppm': 370,
  'free_cn_cil_tank_3_ppm': 395,
  'free_cn_cil_tank_4_ppm': 370,
  'free_cn_cil_tank_5_ppm': 360,
  'free_cn_cil_tank_6_ppm': 360,
  'free_cn_cil_tank_7_ppm': 385,
  'free_cn_cil_tank_8_ppm': 370,
  'free_cn_cil_tank_9_ppm': 370,
  'do_leach_tank_1_ppm': 22,
  'do_leach_tank_2_ppm': 10.31,
  'do_cil_tank_1_ppm': 8.48,
  'do_cil_tank_2_ppm': 8.49,
  'do_cil_tank_3_ppm': 8.7,
  'do_cil_tank_4_ppm': 7.8,
  'do_cil_tank_5_pp

## Optional refresh historical dataset

Interim step to read in original Historical dataset from backup prior to appending new data

In [106]:
# Path to the backup historical data parquet file
parquet_path = Path("historical_data_check.parquet")

# Read in the parquet file
df_historical = pd.read_parquet(parquet_path)

# Check the shape and columns
print(f"Shape: {df_historical.shape}")

# Check the date range
print(f"Date range: {df_historical['date'].min()} to {df_historical['date'].max()}")

# Convert rows to JSON records
hist_rows = df_to_json_records(df_historical)

# Check the first few records
print(json.dumps(hist_rows[:2], indent=2))

DRY_RUN = False
TAG = "historical_base_restore"
MODE = "replace"

if DRY_RUN:
	log("Dry run enabled — no upload performed.")
else:
	base = BASE_URL.rstrip("/")

	if not base:
		raise ValueError("BASE_URL must not be empty.")
	if not CUSTOMER or not SITE:
		raise ValueError("CUSTOMER and SITE must not be empty.")

	headers = {}

	if ADMIN_TOKEN.strip():
		headers["X-Admin-Token"] = ADMIN_TOKEN.strip()

	session = make_session_with_retry(total=3, backoff=0.8)

	total = len(hist_rows)
	sent = 0
	req_mode = MODE.lower()

	log(f"Uploading {total} rows in batches of {BATCH_SIZE} (mode={req_mode})...")

	while sent < total:
		batch_rows = hist_rows[sent : sent + BATCH_SIZE]
		resp = post_rows(
			session=session,
			base=base,
			headers=headers,
			customer=CUSTOMER,
			site=SITE,
			tag=TAG,
			row_batch=batch_rows,
			req_mode=req_mode,
		)
		log(f"Sent rows {sent}..{sent + len(batch_rows)} -> OK {resp}")
		sent += len(batch_rows)
		req_mode = "append"

	log("Upload complete.")

Shape: (610, 137)
Date range: 2024-06-01 to 2026-01-31
[
  {
    "date": "2024-06-01",
    "percent_solids": 51.04166666666666,
    "wad_cn_tailings_ppm": 312.0,
    "ph_leach_tank_01": 11.46,
    "ph_leach_tank_02": 11.47,
    "ph_cil_tank_01": 11.395,
    "ph_cil_tank_02": 11.25,
    "ph_cil_tank_03": 11.24,
    "ph_cil_tank_04": 11.3,
    "ph_cil_tank_05": 11.24,
    "ph_cil_tank_06": 11.26,
    "ph_cil_tank_07": 11.285,
    "ph_cil_tank_08": 11.34,
    "ph_cil_tank_09": 11.305,
    "free_cn_leach_tank_1_ppm": 485.0,
    "free_cn_leach_tank_2_ppm": 470.0,
    "free_cn_cil_tank_1_ppm": 450.0,
    "free_cn_cil_tank_2_ppm": 435.0,
    "free_cn_cil_tank_3_ppm": 425.0,
    "free_cn_cil_tank_4_ppm": 425.0,
    "free_cn_cil_tank_5_ppm": 450.0,
    "free_cn_cil_tank_6_ppm": 435.0,
    "free_cn_cil_tank_7_ppm": 405.0,
    "free_cn_cil_tank_8_ppm": 385.0,
    "free_cn_cil_tank_9_ppm": 375.0,
    "do_leach_tank_1_ppm": null,
    "do_leach_tank_2_ppm": 0.0,
    "do_cil_tank_1_ppm": 0.0,
    "do

## Optional upload to LeachIT API

This cell uploads the extracted history in batches. The first batch uses the configured mode (append or replace), while all later batches are forced to append, matching the original program logic. Leave `DRY_RUN = True` while testing.

In [107]:
DRY_RUN = False
TAG = "bulk_upload"

# Upload behaviour
MODE = "append"          # "append" or "replace"

if DRY_RUN:
	log("Dry run enabled — no upload performed.")
else:
	base = BASE_URL.rstrip("/")

	if not base:
		raise ValueError("BASE_URL must not be empty.")
	if not CUSTOMER or not SITE:
		raise ValueError("CUSTOMER and SITE must not be empty.")

	headers = {}

	if ADMIN_TOKEN.strip():
		headers["X-Admin-Token"] = ADMIN_TOKEN.strip()

	session = make_session_with_retry(total=3, backoff=0.8)

	total = len(rows)
	sent = 0
	req_mode = MODE.lower()

	log(f"Uploading {total} rows in batches of {BATCH_SIZE} (mode={req_mode})...")

	while sent < total:
		batch_rows = rows[sent : sent + BATCH_SIZE]
		resp = post_rows(
			session=session,
			base=base,
			headers=headers,
			customer=CUSTOMER,
			site=SITE,
			tag=TAG,
			row_batch=batch_rows,
			req_mode=req_mode,
		)
		log(f"Sent rows {sent}..{sent + len(batch_rows)} -> OK {resp}")
		sent += len(batch_rows)
		req_mode = "append"

	log("Upload complete.")

Uploading 28 rows in batches of 5000 (mode=append)...
Sent rows 0..28 -> OK {'ok': True, 'siteId': 'shanta-new-luika', 'version': 'historical_20260325T091216', 'rows': 638, 'columns': ['date', 'percent_solids', 'wad_cn_tailings_ppm', 'ph_leach_tank_01', 'ph_leach_tank_02', 'ph_cil_tank_01', 'ph_cil_tank_02', 'ph_cil_tank_03', 'ph_cil_tank_04', 'ph_cil_tank_05', 'ph_cil_tank_06', 'ph_cil_tank_07', 'ph_cil_tank_08', 'ph_cil_tank_09', 'free_cn_leach_tank_1_ppm', 'free_cn_leach_tank_2_ppm', 'free_cn_cil_tank_1_ppm', 'free_cn_cil_tank_2_ppm', 'free_cn_cil_tank_3_ppm', 'free_cn_cil_tank_4_ppm', 'free_cn_cil_tank_5_ppm', 'free_cn_cil_tank_6_ppm', 'free_cn_cil_tank_7_ppm', 'free_cn_cil_tank_8_ppm', 'free_cn_cil_tank_9_ppm', 'do_leach_tank_1_ppm', 'do_leach_tank_2_ppm', 'do_cil_tank_1_ppm', 'do_cil_tank_2_ppm', 'do_cil_tank_3_ppm', 'do_cil_tank_4_ppm', 'do_cil_tank_5_ppm', 'do_cil_tank_6_ppm', 'do_cil_tank_7_ppm', 'do_cil_tank_8_ppm', 'do_cil_tank_9_ppm', 'carbon_concentration_cil_tank_1_gpl', 